OPEN NEW COLAB & LOAD DATA
What we are doing

Load outputs from previous phases.

Why

Models should never use raw data.

In [ ]:
import pandas as pd

customer_features = pd.read_csv("/content/customer_features_phase2.csv")
claims_features = pd.read_csv("/content/claims_risk_features_phase3.csv")
churn_features = pd.read_csv("/content/churn_features_phase4.csv")


PREPARE DATA FOR CHURN MODEL
What we are doing

Select only the columns needed to predict churn and make them model-ready.

In [ ]:
churn_model_df = churn_features[
    [
        "TENURE_YEARS",
        "AGE_BAND",
        "INCOME_BAND",
        "PREMIUM_BAND",
        "CREDIT_RISK",
        "FAMILY_STATUS",
        "HOME_STATUS",
        "Churn"
    ]
]


In [ ]:
churn_model_df.head()


,TENURE_YEARS,AGE_BAND,INCOME_BAND,PREMIUM_BAND,CREDIT_RISK,FAMILY_STATUS,HOME_STATUS,Churn
0,3.983562,Senior,Low,Low,Good_Credit,Has_Children,Home_Owner,0
1,4.917808,Elder,Low,Medium,Poor_Credit,No_Children,Home_Owner,0
2,13.200000,Senior,Low,Medium,Poor_Credit,No_Children,Home_Owner,0
3,0.356164,Senior,High,Medium,Good_Credit,Has_Children,Home_Owner,1
4,16.153425,Senior,Medium,Low,Good_Credit,Has_Children,Home_Owner,0


ENCODE CATEGORICAL VARIABLES
What we are doing

Convert text columns (Senior, Low, Home_Owner…) into numbers.

Why

Models cannot understand text, only numbers.
Encoding lets the model measure impact of each group.

In [ ]:
churn_encoded = pd.get_dummies(
    churn_model_df,
    drop_first=True
)


In [ ]:
churn_encoded.head()


,TENURE_YEARS,Churn,AGE_BAND_Mid,AGE_BAND_Senior,AGE_BAND_Young,INCOME_BAND_Low,INCOME_BAND_Medium,PREMIUM_BAND_Low,PREMIUM_BAND_Medium,CREDIT_RISK_Poor_Credit,FAMILY_STATUS_No_Children,HOME_STATUS_Renter
0,3.983562,0,False,True,False,True,False,True,False,False,False,False
1,4.917808,0,False,False,False,True,False,False,True,True,True,False
2,13.200000,0,False,True,False,True,False,False,True,True,True,False
3,0.356164,1,False,True,False,False,False,False,True,False,False,False
4,16.153425,0,False,True,False,False,True,True,False,False,False,False


TRAIN / TEST SPLIT

What we are doing:
Split data into training and testing sets.

Why:
To check if the model works on new customers, not just memorise old ones.

In [ ]:
from sklearn.model_selection import train_test_split

X = churn_encoded.drop("Churn", axis=1)
y = churn_encoded["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)


In [ ]:
X_train.shape, X_test.shape


((35000, 11), (15000, 11))

BUILD LOGISTIC REGRESSION MODEL

What we are doing:
Train a churn prediction model.

Why:
Logistic regression is:

explainable

business-friendly

perfect for churn problems

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)


LogisticRegression(max_iter=1000)

In [ ]:
model.coef_


array([[-0.11782696, -0.00702169,  0.12524731,  0.00401123, -0.05973146,
        -0.0390104 , -0.12032263, -0.13724639,  0.05435774, -0.09722727,
         0.11202429]])

EVALUATE THE MODEL
What we are doing

Check how well the model predicts churn.

Why

A model is useless if we don’t know how it performs.

In [ ]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test)
accuracy_score(y_test, y_pred)


0.8787333333333334

CONFUSION MATRIX
What we are doing

See where the model is right and wrong.

Why

Churn is imbalanced (few churners), so we must see:

How many churners we actually catch

In [ ]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_test, y_pred)


array([[13181,     0],
       [ 1819,     0]])

FIX CLASS IMBALANCE
Step 1 — Use class_weight='balanced'

What we are doing:
Tell the model: “Churn is rare but important.”

Why:
Missing churners is worse than flagging extra ones.

In [ ]:
from sklearn.linear_model import LogisticRegression

model_balanced = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

model_balanced.fit(X_train, y_train)


LogisticRegression(class_weight='balanced', max_iter=1000)

In [ ]:
y_pred_bal = model_balanced.predict(X_test)


New Confusion Matrix

By correcting for class imbalance, the model successfully identified a majority of churners, enabling proactive retention strategies instead of reactive analysis.

In [ ]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_test, y_pred_bal)


array([[7759, 5422],
       [ 599, 1220]])

EVALUATE WITH PRECISION & RECALL

Accuracy is meaningless here.
We care about Recall and Precision.

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_bal))


              precision    recall  f1-score   support

           0       0.93      0.59      0.72     13181
           1       0.18      0.67      0.29      1819

    accuracy                           0.60     15000
   macro avg       0.56      0.63      0.50     15000
weighted avg       0.84      0.60      0.67     15000



EXTRACT CHURN DRIVERS (MOST IMPORTANT)
What we are doing

See which factors increase or reduce churn.

In [ ]:
import pandas as pd
import numpy as np

coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model_balanced.coef_[0]
}).sort_values(by="Coefficient", ascending=False)

coef_df


,Feature,Coefficient
2,AGE_BAND_Senior,0.101425
10,HOME_STATUS_Renter,0.100568
8,CREDIT_RISK_Poor_Credit,0.048496
3,AGE_BAND_Young,0.013606
1,AGE_BAND_Mid,-0.036790
5,INCOME_BAND_Medium,-0.048250
4,INCOME_BAND_Low,-0.068873
9,FAMILY_STATUS_No_Children,-0.089904
0,TENURE_YEARS,-0.101256
6,PREMIUM_BAND_Low,-0.116388


FACTORS THAT REDUCE CHURN RISK
Feature	Meaning
TENURE_YEARS (–0.10)	Loyalty strongly reduces churn
PREMIUM_BAND_Medium (–0.14)	Core revenue customers stay longer
PREMIUM_BAND_Low (–0.12)	Surprisingly stable low-premium base
FAMILY_STATUS_No_Children (–0.09)	Slightly more stable than expected
INCOME_BAND_Low (–0.07)	Lower income ≠ higher churn (important!)

Insight:
Tenure and pricing position matter more than income alone.

Churn is driven more by customer stability and tenure than income. Renters, seniors, and customers with poor credit are the highest churn risk, while loyal and core-premium customers are the most stable.

In [ ]:
churn_features["CHURN_PROB"] = model_balanced.predict_proba(X)[:, 1]


In [ ]:
churn_features[["CHURN_PROB"]].head()


,CHURN_PROB
0,0.584559
1,0.520825
2,0.342123
3,0.680690
4,0.295228


In [ ]:
excel_input = churn_features[
    ["TENURE_BAND", "INCOME_BAND", "PREMIUM_BAND", "CHURN_PROB", "curr_ann_amt"]
]

excel_input.to_csv("/content/excel_retention_input.csv", index=False)


In [ ]:
from google.colab import files
files.download("/content/excel_retention_input.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>